<h2>Joining Sequence/metadata Information with Interaction datset</h2>
<h6>We keep only interaction with avaiable sequences in both Interacting RNAs</h6>

In [ ]:
## import pandas as pd

# load interactions
df_filt = pd.read_pickle("interactions_processed_noRNAcentralIDs.pkl")
rna_entities_ncRNA = pd.read_pickle("rna_entities_ncRNA.pkl")
rna_entities_mRNA = pd.read_pickle("rna_entities_mRNA.pkl") # ADD FILE-NAME

# ----------------------------
# Helper: canonical entity key
# ----------------------------
def make_entity_key(symbol, category, species, raw_id_list):
    """
    Key: (symbol:str, category:str, species:str, sorted_tuple_of_ids)
    Works if raw_id_list is list, tuple, or other iterable; returns () if empty.
    """
    # normalize inputs to strings
    symbol_s = "" if pd.isna(symbol) else str(symbol)
    category_s = "" if pd.isna(category) else str(category)
    species_s = "" if pd.isna(species) else str(species)

    # normalize raw_id_list to iterable of strings
    if raw_id_list is None:
        ids = []
    elif isinstance(raw_id_list, (list, tuple, set)):
        ids = [str(x) for x in raw_id_list if x is not None]
    else:
        # fallback: single scalar
        ids = [str(raw_id_list)]

    id_tuple = tuple(sorted(ids))
    return (symbol_s, category_s, species_s, id_tuple)

# ----------------------------
# Join rna_entities_ncRNA and rna_entities_mRNA into a single dataframe
# ----------------------------
rna_entities_final = pd.concat([rna_entities_ncRNA, rna_entities_mRNA], ignore_index=True)

# keep only rows that have at least one sequence (list with length>0)
rna_entities_final = rna_entities_final[
    rna_entities_final['sequences'].apply(lambda x: isinstance(x, list) and len(x) > 0)
].reset_index(drop=True)

print("RNA entities with sequences (ncRNA + mRNA):", len(rna_entities_final))

# ----------------------------
# Add canonical entity_key to rna_entities_final
# ----------------------------
# Accept different column names for raw id key: prefer 'Raw_ID_key' then 'Raw_IDs' then 'Raw_IDs'
raw_id_col_candidates = ['Raw_ID_key', 'Raw_IDs', 'Raw_IDs_list', 'Raw_IDs']  # fallback list
def pick_raw_id_col(df):
    for c in raw_id_col_candidates:
        if c in df.columns:
            return c
    return None

raw_col = pick_raw_id_col(rna_entities_final)
if raw_col is None:
    # if none, try to fall back to 'ID_buckets' -> cannot derive Raw_ID_key; require user to supply
    raise RuntimeError("No raw ID column found in rna_entities_final (expected Raw_ID_key or similar).")

rna_entities_final['entity_key'] = rna_entities_final.apply(
    lambda r: make_entity_key(
        r.get('Symbol'),
        r.get('Category'),
        r.get('Species'),
        r.get(raw_col)
    ),
    axis=1
)

# ----------------------------
# Build rna_map dataframe for merges
# ----------------------------
rna_map = rna_entities_final[[
    'entity_key', 'sequences', 'GO', 'Rfam', 'ID_buckets'
]].copy()

# ----------------------------
# Build composite keys for interactions (both interactors)
# ----------------------------
df_filt_merged = df_filt.copy()

# ensure Raw_ID lists exist in df_filt_merged with expected names
# expected: 'Raw_ID1_list' and 'Raw_ID2_list' based on earlier pipeline
if 'Raw_ID1_list' not in df_filt_merged.columns or 'Raw_ID2_list' not in df_filt_merged.columns:
    raise RuntimeError("df_filt missing Raw_ID1_list or Raw_ID2_list columns.")

df_filt_merged['key1'] = df_filt_merged.apply(
    lambda r: make_entity_key(
        r.get('Interactor1.Symbol'),
        r.get('Category1'),
        r.get('Species1_TaxID'),
        r.get('Raw_ID1_list')
    ),
    axis=1
)

df_filt_merged['key2'] = df_filt_merged.apply(
    lambda r: make_entity_key(
        r.get('Interactor2.Symbol'),
        r.get('Category2'),
        r.get('Species2_TaxID'),
        r.get('Raw_ID2_list')
    ),
    axis=1
)

# ----------------------------
# Merge sequences/metadata for interactor1
# ----------------------------
# right-side columns will be renamed to avoid collision
rna_map_1 = rna_map.rename(columns={
    'sequences': 'sequences_1',
    'GO': 'GO_1',
    'Rfam': 'Rfam_1',
    'ID_buckets': 'ID_buckets_1'
})

df_filt_merged = df_filt_merged.merge(
    rna_map_1,
    left_on='key1',
    right_on='entity_key',
    how='left'
)

# drop extra entity_key column produced by merge
if 'entity_key' in df_filt_merged.columns:
    df_filt_merged.drop(columns=['entity_key'], inplace=True)

# ----------------------------
# Merge sequences/metadata for interactor2
# ----------------------------
rna_map_2 = rna_map.rename(columns={
    'sequences': 'sequences_2',
    'GO': 'GO_2',
    'Rfam': 'Rfam_2',
    'ID_buckets': 'ID_buckets_2'
})

df_filt_merged = df_filt_merged.merge(
    rna_map_2,
    left_on='key2',
    right_on='entity_key',
    how='left'
)

if 'entity_key' in df_filt_merged.columns:
    df_filt_merged.drop(columns=['entity_key'], inplace=True)

# ----------------------------
# Normalize merged column names to final names
# ----------------------------
# create consistent column names used downstream
df_filt_merged = df_filt_merged.rename(columns={
    'sequences_1': 'seq1_list',
    'GO_1': 'GO1',
    'Rfam_1': 'Rfam1',
    'ID_buckets_1': 'ID_buckets1',
    'sequences_2': 'seq2_list',
    'GO_2': 'GO2',
    'Rfam_2': 'Rfam2',
    'ID_buckets_2': 'ID_buckets2'
})

# If any missing sequence columns were not created by merge, ensure they exist
for c in ['seq1_list', 'seq2_list', 'GO1', 'GO2', 'Rfam1', 'Rfam2', 'ID_buckets1', 'ID_buckets2']:
    if c not in df_filt_merged.columns:
        df_filt_merged[c] = pd.NA

# ----------------------------
# Step: drop interactions without sequences on either side
# ----------------------------
has_seq1 = df_filt_merged['seq1_list'].apply(lambda x: isinstance(x, list) and len(x) > 0)
has_seq2 = df_filt_merged['seq2_list'].apply(lambda x: isinstance(x, list) and len(x) > 0)

df_filt_final = df_filt_merged[has_seq1 & has_seq2].reset_index(drop=True)

print(f"Total interactions before filtering: {len(df_filt)}")
print(f"Total interactions after filtering: {len(df_filt_final)}")


In [99]:
df_filt_final.to_pickle("interactions_with_sequences_RNA_RNA_10-03-2026.pkl")

In [100]:
import pandas as pd

# Collect all entity_keys present in the filtered interactions
keys_retained = pd.concat([
    df_filt_final['key1'],
    df_filt_final['key2']
]).drop_duplicates()

print(f"Unique RNA entities retained: {len(keys_retained)}")


Unique RNA entities retained: 11929


In [101]:
def count_interactions_under_length(df, max_len):

    kept_rows = []

    for _, row in df.iterrows():

        seq1 = row['seq1_list']
        seq2 = row['seq2_list']

        if not isinstance(seq1, list) or not isinstance(seq2, list):
            continue

        valid_1 = any(len(s) <= max_len for s in seq1)
        valid_2 = any(len(s) <= max_len for s in seq2)

        if valid_1 and valid_2:
            kept_rows.append(row)

    df_kept = pd.DataFrame(kept_rows)

    total_rows = len(df_kept)

    print(f"\nMax length threshold: {max_len} nt")
    print(f"Total interactions: {total_rows}")

    # ----------------------------
    # Row-level coverage
    # ----------------------------
    count_with_seq = (
        df_kept['seq1_list'].apply(lambda x: isinstance(x, list) and len(x) > 0) &
        df_kept['seq2_list'].apply(lambda x: isinstance(x, list) and len(x) > 0)
    ).sum()

    def has_valid_list(x):
        return isinstance(x, list) and any(pd.notna(v) and v != "" for v in x)
    
    # Masks per side
    go1 = df_kept['GO1'].apply(has_valid_list)
    go2 = df_kept['GO2'].apply(has_valid_list)
    
    rfam1 = df_kept['Rfam1'].apply(has_valid_list)
    rfam2 = df_kept['Rfam2'].apply(has_valid_list)
    
    # BOTH sides annotated
    count_go_both = (go1 & go2).sum()
    count_rfam_both = (rfam1 & rfam2).sum()
    
    # AT LEAST ONE side annotated
    count_with_go = (go1 | go2).sum()
    count_with_rfam = (rfam1 | rfam2).sum()

    print(f"Interactions with sequences (both sides): {count_with_seq} ({count_with_seq/total_rows:.2%})")
    print(f"Interactions with GO terms (both sides): {count_go_both} ({count_go_both/total_rows:.2%})")
    print(f"Interactions with GO terms (one side at least): {count_with_go} ({count_with_go/total_rows:.2%})")
    print(f"Interactions with Rfam IDs (both sides): {count_rfam_both} ({count_rfam_both/total_rows:.2%})")
    print(f"Interactions with Rfam IDs (one side at least): {count_with_rfam} ({count_with_rfam/total_rows:.2%})")

    # ----------------------------
    # Sequence-level statistics
    # ----------------------------

    total_sequences = 0
    all_sequences = set()

    for col in ['seq1_list', 'seq2_list']:
        for seq_list in df_kept[col]:
            if isinstance(seq_list, list):
                total_sequences += len(seq_list)
                all_sequences.update(seq_list)

    total_unique_sequences = len(all_sequences)

    print(f"Total sequences (with redundancy): {total_sequences}")
    print(f"Total unique sequences: {total_unique_sequences}")

    if total_sequences > 0:
        redundancy = 1 - total_unique_sequences / total_sequences
        print(f"Redundancy ratio: {redundancy:.2%}")
    else:
        print("Redundancy ratio: NA")

    keys_retained = pd.concat([
        df_kept['key1'],
        df_kept['key2']
    ]).drop_duplicates()
    
    print(f"Unique RNA entities retained: {len(keys_retained)}")

    return df_kept



In [102]:
df_1000000 = count_interactions_under_length(df_filt_final, 1000000)


Max length threshold: 1000000 nt
Total interactions: 82171
Interactions with sequences (both sides): 82171 (100.00%)
Interactions with GO terms (both sides): 6314 (7.68%)
Interactions with GO terms (one side at least): 18164 (22.11%)
Interactions with Rfam IDs (both sides): 6584 (8.01%)
Interactions with Rfam IDs (one side at least): 21230 (25.84%)
Total sequences (with redundancy): 439193
Total unique sequences: 26994
Redundancy ratio: 93.85%
Unique RNA entities retained: 11929


In [104]:
df_10000 = count_interactions_under_length(df_filt_final, 10000)
df_4096 = count_interactions_under_length(df_filt_final, 4096)
df_2048 = count_interactions_under_length(df_filt_final, 2048)
df_1024 = count_interactions_under_length(df_filt_final, 1024)


Max length threshold: 10000 nt
Total interactions: 81458
Interactions with sequences (both sides): 81458 (100.00%)
Interactions with GO terms (both sides): 6293 (7.73%)
Interactions with GO terms (one side at least): 17866 (21.93%)
Interactions with Rfam IDs (both sides): 6537 (8.02%)
Interactions with Rfam IDs (one side at least): 20671 (25.38%)
Total sequences (with redundancy): 436769
Total unique sequences: 26819
Redundancy ratio: 93.86%
Unique RNA entities retained: 11849

Max length threshold: 4096 nt
Total interactions: 75458
Interactions with sequences (both sides): 75458 (100.00%)
Interactions with GO terms (both sides): 6113 (8.10%)
Interactions with GO terms (one side at least): 15832 (20.98%)
Interactions with Rfam IDs (both sides): 6349 (8.41%)
Interactions with Rfam IDs (one side at least): 18417 (24.41%)
Total sequences (with redundancy): 409189
Total unique sequences: 25054
Redundancy ratio: 93.88%
Unique RNA entities retained: 10927

Max length threshold: 2048 nt
Tota